Preparación de Ambiente

In [0]:
%python
dbutils.widgets.removeAll()

In [0]:
dbutils.widgets.text("storageLocation","adlscontenedor")

In [0]:
storageLocation = dbutils.widgets.get("storageLocation")

EXTERNAL LOCATIONS (extl)


In [0]:
%sql
CREATE EXTERNAL LOCATION IF NOT EXISTS `extl-catalog`
URL  'abfss://catalog-mt@${storageLocation}.dfs.core.windows.net/'
WITH (STORAGE CREDENTIAL `credential` )
COMMENT 'Ubicación externa para las tablas de unit-catalog del data lake de retail';

In [0]:
%sql
CREATE EXTERNAL LOCATION IF NOT EXISTS `extl-raw`
URL 'abfss://raw@${storageLocation}.dfs.core.windows.net/'
WITH (STORAGE CREDENTIAL `credential` )
COMMENT 'Datos raw de empresa retail';

In [0]:
%sql
CREATE EXTERNAL LOCATION IF NOT EXISTS `extl-bronze`
URL 'abfss://bronze@${storageLocation}.dfs.core.windows.net/'
WITH (STORAGE CREDENTIAL `credential` )
COMMENT 'Datos Bronze de empresa retail';

In [0]:
%sql
CREATE EXTERNAL LOCATION IF NOT EXISTS `extl-silver`
URL 'abfss://silver@${storageLocation}.dfs.core.windows.net/'
WITH (STORAGE CREDENTIAL `credential` )
COMMENT 'Datos Silver de la empresa olist';

In [0]:
%sql
CREATE EXTERNAL LOCATION IF NOT EXISTS `extl-golden`
URL 'abfss://golden@${storageLocation}.dfs.core.windows.net/'
WITH (STORAGE CREDENTIAL credential)
COMMENT 'Datos Golden de empresa retial'

CATÁLOGO

In [0]:
%sql
DROP CATALOG IF EXISTS catalog_dev CASCADE;
     

In [0]:
%sql
CREATE CATALOG IF NOT EXISTS catalog_dev
MANAGED LOCATION 'abfss://catalog-mt@${storageLocation}.dfs.core.windows.net/'
COMMENT 'Catalogo para la arquitectura medallion del ambiente de dev';

ESQUEMAS

In [0]:
%python
dbutils.fs.rm(f"abfss://bronze@{storageLocation}.dfs.core.windows.net/",True)
dbutils.fs.rm(f"abfss://silver@{storageLocation}.dfs.core.windows.net/",True)
dbutils.fs.rm(f"abfss://golden@{storageLocation}.dfs.core.windows.net/",True)

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS catalog_dev.raw;
CREATE SCHEMA IF NOT EXISTS catalog_dev.bronze;
CREATE SCHEMA IF NOT EXISTS catalog_dev.silver;
CREATE SCHEMA IF NOT EXISTS catalog_dev.golden;

## TABLAS BRONZE

In [0]:
%sql
CREATE TABLE IF NOT EXISTS catalog_dev.bronze.ventas (
ID_ORDER STRING,
STORE STRING,
REG INT,
DATE DATE,
ID_ITEM STRING,
QTY INT,
PRICE DECIMAL(12,2),
DISCOUNT DECIMAL(12,2),
ID_SELLER STRING,
STATUS STRING,
INGESTION_DATE TIMESTAMP
)
USING DELTA
LOCATION 'abfss://bronze@${storageLocation}.dfs.core.windows.net/ventas';

In [0]:
%sql
CREATE TABLE IF NOT EXISTS catalog_dev.bronze.productos (
ID_ITEM STRING,
PRODUCT STRING,
CATEGORY STRING,
INGESTION_DATE TIMESTAMP
)
USING DELTA
LOCATION 'abfss://bronze@${storageLocation}.dfs.core.windows.net/productos';


## TABLAS SILVER

In [0]:
%sql
CREATE TABLE IF NOT EXISTS catalog_dev.silver.ventas_productos_categorias (
  ID_ARTICULO STRING,
  PRODUCTO STRING,
  CATEGORIA STRING,
  ID_PEDIDO STRING,
  TIENDA STRING,
  NUM_LINEA INT,
  FECHA DATE,
  CANTIDAD INT,
  PRECIO DOUBLE,
  DESCUENTO INT,
  ID_VENDEDOR STRING,
  ETIQUETA_DESCUENTO STRING
)
USING DELTA
LOCATION 'abfss://silver@${storageLocation}.dfs.core.windows.net/ventas_productos_categorias';

CREATE TABLE IF NOT EXISTS catalog_dev.silver.ventas_test (
  ID_ARTICULO STRING,
  PRODUCTO STRING,
  CATEGORIA STRING,
  ID_PEDIDO STRING,
  TIENDA STRING,
  NUM_LINEA INT,
  FECHA DATE,
  CANTIDAD INT,
  PRECIO DOUBLE,
  DESCUENTO INT,
  ID_VENDEDOR STRING,
  ETIQUETA_DESCUENTO STRING
)
USING DELTA
LOCATION 'abfss://silver@${storageLocation}.dfs.core.windows.net/ventas_test';

Tablas Gold

In [0]:
%sql
CREATE TABLE IF NOT EXISTS catalog_dev.golden.ventas_diarias_tienda (
  TIENDA                    STRING,
  FECHA                     DATE,
  NUM_PEDIDOS               INT,
  TOTAL_UNIDADES            INT,
  INGRESO_BRUTO             DOUBLE,
  INGRESO_NETO              DOUBLE,
  DESCUENTO_MEDIO_PCT       DOUBLE,
  NUM_LINEAS_CON_DESCUENTO  INT,
  NUM_LINEAS_TOTAL          INT,
  PCT_LINEAS_CON_DESCUENTO  DOUBLE,
  FECHA_ACTUALIZACION       TIMESTAMP
)
USING DELTA
LOCATION 'abfss://golden@${storageLocation}.dfs.core.windows.net/ventas_diarias_tienda';

In [0]:
%sql
CREATE TABLE IF NOT EXISTS catalog_dev.golden.ventas_categoria_mes (
  CATEGORIA                 STRING,
  ANIO                      INT,
  MES                       INT,
  NUM_PEDIDOS               INT,
  NUM_ARTICULOS             INT,
  TOTAL_UNIDADES            INT,
  NUM_LINEAS_TOTAL          INT,
  INGRESO_BRUTO             DOUBLE,
  INGRESO_NETO              DOUBLE,
  TICKET_PROMEDIO           DOUBLE,
  PRECIO_PROMEDIO           DOUBLE,
  NUM_LINEAS_CON_DESCUENTO  INT,
  PCT_LINEAS_CON_DESCUENTO  DOUBLE,
  DESCUENTO_MEDIO_PCT       DOUBLE,
  NUM_PRECIO_NORMAL         INT,
  NUM_PROMOCION             INT,
  NUM_BONO_EMPRESARIAL      INT,
  NUM_REGALO                INT,
  FECHA_ACTUALIZACION       TIMESTAMP
)
USING DELTA
LOCATION 'abfss://golden@${storageLocation}.dfs.core.windows.net/ventas_categoria_mes';